# HKH Glacial Lake Statistics (2020 vs 2025)

This notebook computes summary statistics for HKH glacial lakes for **2020** and **2025**, as discussed in the IGARSS paper plan (Points 1 and 2):

**Point 1 – Basic 2025 inventory stats**
- Total number and total area of lakes (2020 & 2025)
- Lake size distribution (area classes)
- Elevation distribution (bands)
- Optional: distribution by region / basin / country (if boundary shapefiles provided)

**Point 2 – Change statistics (2020 → 2025)**
- Change in total number and total area (ΔN, ΔA, PAC, APAC)
- New vs vanished vs stable lakes
- Expanding vs shrinking lakes (per-lake area change)
- Change by size class
- Change by elevation band and region

Fill in the paths in the configuration cell and run the notebook top to bottom.

In [ ]:
# === Imports ===
import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon
import numpy as np

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 50)

In [ ]:
# === Configuration ===

# Required shapefiles
LAKES_2020_SHP = "path/to/hkh_lakes_2020.shp"  # <-- update
LAKES_2025_SHP = "path/to/hkh_lakes_2025.shp"  # <-- update

# Optional boundary layers for regional stats (set to None if not available)
REGION_BOUNDS_SHP = None      # e.g. HKH subregions (Karakoram, Hindu Kush, etc.)
BASIN_BOUNDS_SHP  = None      # e.g. major river basins
COUNTRY_BOUNDS_SHP = None     # e.g. country polygons

# Name of area column (will be created if not present), in km^2
AREA_COL = "area_km2"

# If your lake shapefiles already have an elevation attribute, put its name here.
# If not, you can later extend the notebook to sample a DEM and fill this column.
LAKE_ELEV_COL = "elev_mean"   # e.g. mean lake surface elevation in meters

# Time period between inventories (years) – used for APAC
DELTA_T_YEARS = 5.0  # 2020 → 2025

# CRS for area calculation (equal-area projection)
AREA_CRS = "EPSG:6933"  # World Equidistant Cylindrical (suitable global equal-area)

In [ ]:
# === Helper functions ===

def load_lakes(path: str, area_crs: str = AREA_CRS, area_col: str = AREA_COL):
    """Load a lake shapefile, project to equal-area CRS and compute area in km^2.

    Parameters
    ----------
    path : str
        Path to shapefile (.shp) or other vector file.
    area_crs : str
        EPSG code or proj string for equal-area CRS.
    area_col : str
        Name of area column to create.
    """
    gdf = gpd.read_file(path)
    
    if gdf.crs is None:
        raise ValueError(f"Input lakes file {path} has no CRS. Please define it before using this notebook.")

    gdf = gdf.to_crs(area_crs)
    gdf[area_col] = gdf.geometry.area / 1e6  # m^2 → km^2
    return gdf


def make_size_classes(area_series: pd.Series):
    """Create lake size classes (km^2) following Li et al. style.

    Modify bins/labels if you want a different scheme.
    """
    bins = [0, 0.01, 0.05, 0.1, 0.5, 1, 5, 10, np.inf]
    labels = [
        "<0.01",
        "0.01–0.05",
        "0.05–0.1",
        "0.1–0.5",
        "0.5–1",
        "1–5",
        "5–10",
        ">10",
    ]
    cat = pd.cut(area_series, bins=bins, labels=labels, right=False, include_lowest=True)
    return cat


def make_elev_bands(elev_series: pd.Series):
    """Create elevation bands (m). Adjust bins as needed for HKH.

    Example bands: 3,000–3,500, 3,500–4,000, ..., 5,500–6,000.
    """
    bins = [3000, 3500, 4000, 4500, 5000, 5500, 6000, np.inf]
    labels = [
        "3000–3500",
        "3500–4000",
        "4000–4500",
        "4500–5000",
        "5000–5500",
        "5500–6000",
        ">6000",
    ]
    cat = pd.cut(elev_series, bins=bins, labels=labels, right=False)
    return cat


def summarize_lakes(gdf: gpd.GeoDataFrame, name: str):
    """Print and return basic summary statistics for a lake inventory."""
    total_n = len(gdf)
    total_area = gdf[AREA_COL].sum()
    print(f"=== {name} lake inventory ===")
    print(f"Number of lakes: {total_n}")
    print(f"Total area (km²): {total_area:.3f}\n")
    return {"N": total_n, "A_km2": total_area}


def size_distribution(gdf: gpd.GeoDataFrame, name: str):
    """Compute lake size distribution (count & area per class)."""
    gdf = gdf.copy()
    gdf["size_class"] = make_size_classes(gdf[AREA_COL])
    grp = gdf.groupby("size_class")[AREA_COL].agg(["count", "sum"]).rename(columns={"count": "n", "sum": "area_km2"})
    grp["fraction_n"] = grp["n"] / grp["n"].sum()
    grp["fraction_area"] = grp["area_km2"] / grp["area_km2"].sum()
    print(f"=== {name} size distribution ===")
    display(grp)
    return grp


def elev_distribution(gdf: gpd.GeoDataFrame, name: str, elev_col: str = LAKE_ELEV_COL):
    """Compute lake elevation distribution (count & area per elevation band).

    Requires an elevation column in meters.
    """
    if elev_col not in gdf.columns:
        print(f"[WARN] Elevation column '{elev_col}' not found. Skipping elevation distribution for {name}.")
        return None

    gdf = gdf.copy()
    gdf["elev_band"] = make_elev_bands(gdf[elev_col])
    grp = gdf.groupby("elev_band")[AREA_COL].agg(["count", "sum"]).rename(columns={"count": "n", "sum": "area_km2"})
    grp["fraction_n"] = grp["n"] / grp["n"].sum()
    grp["fraction_area"] = grp["area_km2"] / grp["area_km2"].sum()
    print(f"=== {name} elevation distribution ===")
    display(grp)
    return grp

In [ ]:
# === Load 2020 and 2025 lake inventories ===

gdf_2020 = load_lakes(LAKES_2020_SHP)
gdf_2025 = load_lakes(LAKES_2025_SHP)

# Add a simple ID if not present (used later for change matching)
if "lake_id" not in gdf_2020.columns:
    gdf_2020["lake_id"] = [f"L2020_{i}" for i in range(len(gdf_2020))]
if "lake_id" not in gdf_2025.columns:
    gdf_2025["lake_id"] = [f"L2025_{i}" for i in range(len(gdf_2025))]

gdf_2020.head(), gdf_2025.head()

In [ ]:
# === 1. Basic inventory statistics (2020 & 2025) ===

summary_2020 = summarize_lakes(gdf_2020, "2020")
summary_2025 = summarize_lakes(gdf_2025, "2025")

# 1.2 Size distribution
size_2020 = size_distribution(gdf_2020, "2020")
size_2025 = size_distribution(gdf_2025, "2025")

# 1.3 Elevation distribution (if elevation column exists)
elev_2020 = elev_distribution(gdf_2020, "2020")
elev_2025 = elev_distribution(gdf_2025, "2025")

In [ ]:
# === Optional: regional statistics (by region / basin / country) ===

def overlay_stats(lakes: gpd.GeoDataFrame, bounds_path: str, bounds_label_col: str, name: str):
    """Overlay lakes with a boundary layer and compute stats per polygon.

    Parameters
    ----------
    lakes : GeoDataFrame
        Lake polygons in AREA_CRS.
    bounds_path : str
        Path to boundary layer (e.g. regions, basins, countries).
    bounds_label_col : str
        Column in boundary shapefile to use as label (e.g. 'NAME').
    name : str
        Label for printing (e.g. 'Region', 'Basin').
    """
    if bounds_path is None:
        print(f"[INFO] No {name} boundary shapefile provided. Skipping.")
        return None

    bounds = gpd.read_file(bounds_path)
    bounds = bounds.to_crs(AREA_CRS)

    # spatial join: lakes -> which polygon they fall in
    joined = gpd.sjoin(lakes, bounds[[bounds_label_col, "geometry"]], how="left", predicate="intersects")

    grp = joined.groupby(bounds_label_col)[AREA_COL].agg(["count", "sum"]).rename(columns={"count": "n", "sum": "area_km2"})
    grp["fraction_n"] = grp["n"] / grp["n"].sum()
    grp["fraction_area"] = grp["area_km2"] / grp["area_km2"].sum()
    print(f"=== {name} statistics ===")
    display(grp.sort_values("area_km2", ascending=False))
    return grp

# Example usage (uncomment and set column names):
# region_stats_2025 = overlay_stats(gdf_2025, REGION_BOUNDS_SHP, bounds_label_col="REGION_NAME", name="Region (2025)")
# basin_stats_2025  = overlay_stats(gdf_2025, BASIN_BOUNDS_SHP,  bounds_label_col="BASIN_NAME",  name="Basin (2025)")
# country_stats_2025 = overlay_stats(gdf_2025, COUNTRY_BOUNDS_SHP, bounds_label_col="COUNTRY", name="Country (2025)")

In [ ]:
# === 2. Change statistics: 2020 → 2025 ===

# 2.1 Change in total number and area
delta_N = summary_2025["N"] - summary_2020["N"]
delta_A = summary_2025["A_km2"] - summary_2020["A_km2"]

PAC = (delta_A / summary_2020["A_km2"]) * 100.0  # Percentage of Area Change
APAC = PAC / DELTA_T_YEARS                            # Annual Percentage of Area Change

print("=== 2020 → 2025 change (global) ===")
print(f"ΔN (number of lakes): {delta_N}")
print(f"ΔA (area, km²): {delta_A:.3f}")
print(f"PAC (%, 2020→2025): {PAC:.2f}%")
print(f"APAC (%, per year): {APAC:.2f}%/yr\n")

In [ ]:
# === 2.2 Match lakes between 2020 and 2025 to detect new/vanished/expanded/shrunk ===

from shapely.ops import unary_union

def match_lakes_by_overlap(gdf_old: gpd.GeoDataFrame,
                           gdf_new: gpd.GeoDataFrame,
                           iou_threshold: float = 0.01):
    """Match lakes between two inventories using intersection-over-union (IoU).

    For each new (2025) lake, find the old (2020) lake with the highest IoU.
    If IoU < iou_threshold, it is considered a *new* lake.

    Returns
    -------
    matches : DataFrame
        One row per new lake with columns:
        ['lake_id_new', 'lake_id_old', 'area_new', 'area_old', 'iou']
    """
    # Ensure same CRS
    g1 = gdf_old.to_crs(AREA_CRS).copy()
    g2 = gdf_new.to_crs(AREA_CRS).copy()

    # Spatial index for old lakes
    sindex = g1.sindex

    records = []
    
    for idx_new, row_new in g2.iterrows():
        geom_new = row_new.geometry
        # candidate old lakes whose bbox intersects new lake
        possible_matches_idx = list(sindex.intersection(geom_new.bounds))
        if not possible_matches_idx:
            # no overlap at all → new lake
            records.append({
                "lake_id_new": row_new["lake_id"],
                "lake_id_old": None,
                "area_new": row_new[AREA_COL],
                "area_old": 0.0,
                "iou": 0.0,
            })
            continue

        best_iou = 0.0
        best_old_id = None
        best_old_area = 0.0

        for idx_old in possible_matches_idx:
            row_old = g1.iloc[idx_old]
            geom_old = row_old.geometry
            inter = geom_new.intersection(geom_old)
            if inter.is_empty:
                continue
            inter_area = inter.area
            union_area = geom_new.union(geom_old).area
            iou = inter_area / union_area if union_area > 0 else 0
            if iou > best_iou:
                best_iou = iou
                best_old_id = row_old["lake_id"]
                best_old_area = row_old[AREA_COL]

        if best_iou < iou_threshold:
            # treat as new lake (no meaningful match)
            best_old_id = None
            best_old_area = 0.0

        records.append({
            "lake_id_new": row_new["lake_id"],
            "lake_id_old": best_old_id,
            "area_new": row_new[AREA_COL],
            "area_old": best_old_area,
            "iou": best_iou,
        })

    matches = pd.DataFrame.from_records(records)
    return matches


matches_2020_2025 = match_lakes_by_overlap(gdf_2020, gdf_2025, iou_threshold=0.01)
matches_2020_2025.head()

In [ ]:
# === 2.2a Classify lakes: new, stable, expanded, shrunk ===

def classify_changes(matches: pd.DataFrame,
                     expansion_threshold: float = 0.25,
                     shrink_threshold: float = -0.25):
    """Classify each 2025 lake based on area change relative to its 2020 match.

    Parameters
    ----------
    expansion_threshold : float
        Relative change (e.g., +0.25 = +25%) above which a lake is 'expanded'.
    shrink_threshold : float
        Relative change (e.g., -0.25 = -25%) below which a lake is 'shrunk'.
    """
    df = matches.copy()

    # relative area change with respect to old area
    df["delta_area"] = df["area_new"] - df["area_old"]
    df["rel_change"] = np.where(df["area_old"] > 0,
                                 df["delta_area"] / df["area_old"],
                                 np.nan)

    def classify_row(row):
        if row["lake_id_old"] is None or (row["area_old"] == 0):
            return "new"
        if np.isnan(row["rel_change"]):
            return "stable"  # conservative
        if row["rel_change"] >= expansion_threshold:
            return "expanded"
        if row["rel_change"] <= shrink_threshold:
            return "shrunk"
        return "stable"

    df["change_class"] = df.apply(classify_row, axis=1)
    return df


changes = classify_changes(matches_2020_2025)

print("=== Change class breakdown (2025 lakes) ===")
print(changes["change_class"].value_counts())

changes.head()

In [ ]:
# Find vanished lakes: present in 2020 but not matched in 2025

matched_old_ids = set(changes["lake_id_old"].dropna())
all_old_ids = set(gdf_2020["lake_id"])
vanished_ids = all_old_ids - matched_old_ids

print(f"Number of vanished lakes (present in 2020, absent in 2025): {len(vanished_ids)}")

In [ ]:
# === 2.3 Change by size class ===

# Already have size distributions size_2020 and size_2025 from earlier.

size_change = size_2020[["n", "area_km2"]].rename(columns={"n": "n_2020", "area_km2": "area_2020_km2"})\
    .join(
        size_2025[["n", "area_km2"]].rename(columns={"n": "n_2025", "area_km2": "area_2025_km2"}),
        how="outer"
    )

size_change = size_change.fillna(0)
size_change["delta_n"] = size_change["n_2025"] - size_change["n_2020"]
size_change["delta_area_km2"] = size_change["area_2025_km2"] - size_change["area_2020_km2"]

print("=== Change by size class (2020 → 2025) ===")
display(size_change)

In [ ]:
# === 2.4 Change by elevation band (if elevation data available) ===

if (LAKE_ELEV_COL in gdf_2020.columns) and (LAKE_ELEV_COL in gdf_2025.columns):
    gdf_2020_elev = gdf_2020.copy()
    gdf_2020_elev["elev_band"] = make_elev_bands(gdf_2020_elev[LAKE_ELEV_COL])

    gdf_2025_elev = gdf_2025.copy()
    gdf_2025_elev["elev_band"] = make_elev_bands(gdf_2025_elev[LAKE_ELEV_COL])

    elev_2020_band = gdf_2020_elev.groupby("elev_band")[AREA_COL].agg(["count", "sum"]).rename(columns={"count": "n_2020", "sum": "area_2020_km2"})
    elev_2025_band = gdf_2025_elev.groupby("elev_band")[AREA_COL].agg(["count", "sum"]).rename(columns={"count": "n_2025", "sum": "area_2025_km2"})

    elev_change = elev_2020_band.join(elev_2025_band, how="outer").fillna(0)
    elev_change["delta_n"] = elev_change["n_2025"] - elev_change["n_2020"]
    elev_change["delta_area_km2"] = elev_change["area_2025_km2"] - elev_change["area_2020_km2"]

    print("=== Change by elevation band (2020 → 2025) ===")
    display(elev_change)
else:
    print(f"[WARN] Elevation column '{LAKE_ELEV_COL}' not available in both inventories. Skipping elevation-band change stats.")